# Medallion Pipeline (Bronze → Silver → Gold)


- Bronze (raw data)
- Silver (clean data)
- Gold (business-ready data)


## 1. Setup

- Paths to the volumes with the raw csv files
- Names of all tables created in this notebook

In [0]:
from pyspark.sql import functions as F

# Folders (volumes) where the raw CSV files were uploaded
CHURN_DIR = "/Volumes/workspace/default/customer_data/raw"
TRANSACTIONS_DIR = "/Volumes/workspace/default/bank_data/raw"

# Churn tables
BRONZE_CHURN = "bronze_churn"
SILVER_CHURN = "silver_churn"
QUARANTINE_CHURN = "quarantine_churn"
GOLD_CHURN_CUSTOMERS = "gold_churn_customers"

# Transaction tables
BRONZE_TRANSACTIONS = "bronze_transactions"
SILVER_TRANSACTIONS = "silver_transactions"
QUARANTINE_TRANSACTIONS = "quarantine_transactions"
GOLD_ACCOUNTS = "gold_accounts"
GOLD_TRANSFERS = "gold_transfers"

print("Churn files:", CHURN_DIR)
print("Transaction files:", TRANSACTIONS_DIR)

## Part A: Churn data

### A.1 Read raw file

- Read churn.csv with all columns as strings (inferSchema=false), so no values get changed while loading
- Check schema, row count and first rows
- Check for duplicate customer IDs

In [0]:
raw_churn = (spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{CHURN_DIR}/churn.csv"))

print("Schema of the raw churn data:")
raw_churn.printSchema()

print("CSV rows:", raw_churn.count())
raw_churn.show(5, truncate=False)

# Checking if a customer appears more than once
print("Customers that appear more than once:")
raw_churn.groupBy("CustomerId").count().filter(F.col("count") > 1).show()

### A.2 Bronze

- Save the raw data unchanged as a delta table
- Add three lineage columns: source file, load time and batch id to be able to see where data comes from

In [0]:
bronze_churn = (raw_churn
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("batch_id", F.lit("initial_load")))

(bronze_churn.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(BRONZE_CHURN))

bronze_churn = spark.table(BRONZE_CHURN)
print("Bronze churn row count:", bronze_churn.count())

### A.3 Silver

Step 1: convert types
- Convert the string columns to numbers with try_cast (invalid values become NULL instead of causing an error)
- Rename columns to snake_case
- Drop RowNumber (only the row position) and Surname (personal data, not needed)

In [0]:
# Stage 1: Convert types and rename columns
typed_churn = bronze_churn.select(
    F.col("CustomerId").alias("customer_id"),
    F.expr("try_cast(CreditScore AS INT)").alias("credit_score"),
    F.col("Geography").alias("geography"),
    F.col("Gender").alias("gender"),
    F.expr("try_cast(Age AS INT)").alias("age"),
    F.expr("try_cast(Tenure AS INT)").alias("tenure"),
    F.expr("try_cast(Balance AS DOUBLE)").alias("balance"),
    F.expr("try_cast(NumOfProducts AS INT)").alias("num_of_products"),
    F.expr("try_cast(HasCrCard AS INT)").alias("has_cr_card"),
    F.expr("try_cast(IsActiveMember AS INT)").alias("is_active_member"),
    F.expr("try_cast(EstimatedSalary AS DOUBLE)").alias("estimated_salary"),
    F.expr("try_cast(Exited AS INT)").alias("exited"),
)

typed_churn.printSchema()
typed_churn.show(5)

Step 2: validation rules
- customer_id is not empty
- credit_score between 300 and 850 (normal credit score), age between 18 and 100
- tenure, balance >= 0, estimated_salary > 0, num_of_products >= 1
- has_cr_card, is_active_member, exited are 0 or 1
- geography and gender have known values
- coalesce(..., False) makes sure rows with NULL values count as invalid

In [0]:
# Stage 2: Validating each row
id_ok = F.col("customer_id").isNotNull() & (F.length(F.trim("customer_id")) > 0)

numbers_ok = (
    F.col("credit_score").between(300, 850)
    & F.col("age").between(18, 100)
    & (F.col("tenure") >= 0)
    & (F.col("balance") >= 0)
    & (F.col("estimated_salary") > 0)
    & (F.col("num_of_products") >= 1)
)

flags_ok = (
    F.col("has_cr_card").isin(0, 1)
    & F.col("is_active_member").isin(0, 1)
    & F.col("exited").isin(0, 1)
)

categories_ok = (
    F.col("geography").isin("France", "Germany", "Spain")
    & F.col("gender").isin("Male", "Female")
)

valid_churn = F.coalesce(id_ok & numbers_ok & flags_ok & categories_ok, F.lit(False))

print("Valid rows:", typed_churn.filter(valid_churn).count())
print("Invalid rows (quarantine):", typed_churn.filter(~valid_churn).count())

Step 3: quarantine
- Invalid rows are saved in a separate table with an error_reason instead of being deleted
- In this dataset all rows are valid, so the table is empty. The checks are still useful for future data loads.

In [0]:
# Stage 3: Sending invalid rows to quarantine
valid_churn_rows = typed_churn.filter(valid_churn)

invalid_churn_rows = (typed_churn.filter(~valid_churn)
    .withColumn("error_reason",
        F.when(~F.coalesce(id_ok, F.lit(False)), "missing customer ID")
         .when(~F.coalesce(numbers_ok, F.lit(False)), "numeric value missing or out of range")
         .when(~F.coalesce(flags_ok, F.lit(False)), "flag is not 0 or 1")
         .when(~F.coalesce(categories_ok, F.lit(False)), "unknown geography or gender")
         .otherwise("other validation failure")))

(invalid_churn_rows.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(QUARANTINE_CHURN))

print("Quarantined rows:", spark.table(QUARANTINE_CHURN).count())
spark.table(QUARANTINE_CHURN).groupBy("error_reason").count().show(truncate=False)

Step 4: duplicates
- Keep one row per customer_id with dropDuplicates
- Save the result as the silver table

In [0]:
# Stage 4: Removing duplicates and saving Silver
silver_churn = valid_churn_rows.dropDuplicates(["customer_id"])

(silver_churn.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(SILVER_CHURN))

print("Silver churn row count:", spark.table(SILVER_CHURN).count())
spark.table(SILVER_CHURN).printSchema()

### A.4 Gold
gold_churn_customers
- Input table for the ML model
- Adds age_group (age bands) and has_zero_balance (1 if balance is 0)
- The original age column is kept as well


In [0]:
gold_churn_customers = (spark.table(SILVER_CHURN)
    .withColumn("age_group",
        F.when(F.col("age") < 30, "18-29")
         .when(F.col("age") < 40, "30-39")
         .when(F.col("age") < 50, "40-49")
         .when(F.col("age") < 60, "50-59")
         .otherwise("60+"))
    .withColumn("has_zero_balance", F.when(F.col("balance") == 0, 1).otherwise(0)))

(gold_churn_customers.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_CHURN_CUSTOMERS))

print("Gold churn customers row count:", spark.table(GOLD_CHURN_CUSTOMERS).count())
spark.table(GOLD_CHURN_CUSTOMERS).show(5)

## Part B: Transaction data

### B.1 Read raw file

- Read bank_transactions.csv with all columns as strings
- Check schema, row count and first rows
- Check for duplicate transaction IDs

In [0]:
raw_transactions = (spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{TRANSACTIONS_DIR}/bank_transactions.csv"))

print("Schema of the raw transaction data:")
raw_transactions.printSchema()

print("CSV rows:", raw_transactions.count())
raw_transactions.show(5, truncate=False)

# Checking if a transaction ID appears more than once
print("Transaction IDs that appear more than once:")
raw_transactions.groupBy("Transaction ID").count().filter(F.col("count") > 1).show()

### B.2 Bronze

- Delta does not allow spaces or brackets in column names (e.g. "Latency (ms)"), so the columns are renamed
- Lineage columns are added before renaming, because _metadata is only available directly after reading the file
- Values stay unchanged as strings

In [0]:
# Original column name -> clean column name
column_names = {
    "Transaction ID": "transaction_id",
    "Sender Account ID": "sender_account_id",
    "Receiver Account ID": "receiver_account_id",
    "Transaction Amount": "transaction_amount",
    "Transaction Type": "transaction_type",
    "Timestamp": "transaction_time",
    "Transaction Status": "transaction_status",
    "Fraud Flag": "fraud_flag",
    "Geolocation (Latitude/Longitude)": "geolocation",
    "Device Used": "device_used",
    "Network Slice ID": "network_slice_id",
    "Latency (ms)": "latency_ms",
    "Slice Bandwidth (Mbps)": "slice_bandwidth_mbps",
    "PIN Code": "pin_code",
}

# Adding the lineage columns first, while the hidden _metadata column is still available
bronze_transactions = (raw_transactions
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("batch_id", F.lit("initial_load")))

# Renaming the columns one by one
for old_name, new_name in column_names.items():
    bronze_transactions = bronze_transactions.withColumnRenamed(old_name, new_name)

(bronze_transactions.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(BRONZE_TRANSACTIONS))

bronze_transactions = spark.table(BRONZE_TRANSACTIONS)
print("Bronze transactions row count:", bronze_transactions.count())
bronze_transactions.printSchema()

### B.3 Silver

Step 1: convert types
- Amount to double, time to timestamp, fraud flag to boolean, network values to int
- Drop pin_code (security code, not needed for analysis)
- Drop geolocation: latitudes and longitudes of 6 cities are randomly combined (e.g. latitude of Los Angeles with longitude of Tokyo), so the locations are not real

In [0]:
# Stage 1: Convert types
typed_transactions = bronze_transactions.select(
    "transaction_id",
    "sender_account_id",
    "receiver_account_id",
    F.expr("try_cast(transaction_amount AS DOUBLE)").alias("transaction_amount"),
    "transaction_type",
    F.expr("try_cast(transaction_time AS TIMESTAMP)").alias("transaction_time"),
    "transaction_status",
    F.expr("try_cast(fraud_flag AS BOOLEAN)").alias("fraud_flag"),
    "device_used",
    "network_slice_id",
    F.expr("try_cast(latency_ms AS INT)").alias("latency_ms"),
    F.expr("try_cast(slice_bandwidth_mbps AS INT)").alias("slice_bandwidth_mbps"),
)

typed_transactions.printSchema()
typed_transactions.show(5, truncate=False)

Step 2: validation rules
- Transaction, sender and receiver IDs are not empty, sender and receiver are different
- Amount > 0, time and fraud flag could be converted
- Latency >= 0, bandwidth > 0
- Type, status and device have known values

In [0]:
# Stage 2: Validating each row
def not_empty(column):
    return F.col(column).isNotNull() & (F.length(F.trim(column)) > 0)

ids_ok = (
    not_empty("transaction_id")
    & not_empty("sender_account_id")
    & not_empty("receiver_account_id")
    & (F.col("sender_account_id") != F.col("receiver_account_id"))
)

values_ok = (
    (F.col("transaction_amount") > 0)
    & F.col("transaction_time").isNotNull()
    & F.col("fraud_flag").isNotNull()
    & (F.col("latency_ms") >= 0)
    & (F.col("slice_bandwidth_mbps") > 0)
)

categories_ok_tx = (
    F.col("transaction_type").isin("Deposit", "Transfer", "Withdrawal")
    & F.col("transaction_status").isin("Success", "Failed")
    & F.col("device_used").isin("Mobile", "Desktop")
)

valid_transaction = F.coalesce(ids_ok & values_ok & categories_ok_tx, F.lit(False))

print("Valid rows:", typed_transactions.filter(valid_transaction).count())
print("Invalid rows (quarantine):", typed_transactions.filter(~valid_transaction).count())

In [0]:
# Stage 3: Sending invalid rows to quarantine
valid_transaction_rows = typed_transactions.filter(valid_transaction)

invalid_transaction_rows = (typed_transactions.filter(~valid_transaction)
    .withColumn("error_reason",
        F.when(~F.coalesce(ids_ok, F.lit(False)), "missing or identical account IDs")
         .when(~F.coalesce(values_ok, F.lit(False)), "value missing or out of range")
         .when(~F.coalesce(categories_ok_tx, F.lit(False)), "unknown type, status or device")
         .otherwise("other validation failure")))

(invalid_transaction_rows.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(QUARANTINE_TRANSACTIONS))

print("Quarantined rows:", spark.table(QUARANTINE_TRANSACTIONS).count())
spark.table(QUARANTINE_TRANSACTIONS).groupBy("error_reason").count().show(truncate=False)

In [0]:
# Stage 4: Removing duplicates and saving Silver
silver_transactions = valid_transaction_rows.dropDuplicates(["transaction_id"])

(silver_transactions.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(SILVER_TRANSACTIONS))

print("Silver transactions row count:", spark.table(SILVER_TRANSACTIONS).count())

### B.4 Gold

- Prepares the data for the graph analysis with GraphFrames
- gold_accounts: every account once (union of senders and receivers, then distinct), column id
- gold_transfers: every transaction as an edge from src (sender) to dst (receiver)
- GraphFrames needs exactly these column names

In [0]:
silver_tx = spark.table(SILVER_TRANSACTIONS)

# Nodes: every account that sent or received money
gold_accounts = (silver_tx.select(F.col("sender_account_id").alias("id"))
    .union(silver_tx.select(F.col("receiver_account_id").alias("id")))
    .distinct())

(gold_accounts.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_ACCOUNTS))

# Edges: every transaction from sender (src) to receiver (dst)
gold_transfers = silver_tx.select(
    F.col("sender_account_id").alias("src"),
    F.col("receiver_account_id").alias("dst"),
    "transaction_id", "transaction_amount", "transaction_type",
    "transaction_time", "transaction_status", "fraud_flag", "device_used",
)

(gold_transfers.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_TRANSFERS))

print("Accounts (nodes):", spark.table(GOLD_ACCOUNTS).count())
print("Transfers (edges):", spark.table(GOLD_TRANSFERS).count())